# Mid Cap Cleaning and Metrics

Run `mid_cap_excel_to_csv.ipynb` first. This notebook cleans the five updated Direct Growth mid-cap NAV CSVs for January 2022 through December 2025, calculates daily returns, running maxima, drawdowns, and fund-level metrics, then saves the cleaned series and results.

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path


In [20]:
funds = {
    "Aditya Birla Sun Life Midcap Fund": "Mid_cap/csv/aditya_mid_cap.csv",
    "Axis Midcap Fund": "Mid_cap/csv/axis_mid_cap.csv",
    "DSP Midcap Fund": "Mid_cap/csv/dsp_mid_cap.csv",
    "HDFC Mid Cap Fund": "Mid_cap/csv/hdfc_mid_cap.csv",
    "Mahindra Manulife Mid Cap Fund": "Mid_cap/csv/mahindra_mid_cap.csv"
}


In [21]:
def clean_nav_data(file_path, fund_name):
    df = pd.read_csv(file_path)
    df.columns = df.columns.astype(str).str.strip()
    df = df[["Net Asset Value", "Date", "Plan", "Option"]].copy()
    df = df.rename(columns={"Net Asset Value": "NAV"})

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["NAV"] = pd.to_numeric(df["NAV"], errors="coerce")
    df["Plan"] = df["Plan"].astype(str).str.strip()
    df["Option"] = df["Option"].astype(str).str.strip()

    # Retain Direct Plan + Growth Option, allowing label variations such as GROWTH.
    df = df[
        df["Plan"].str.contains("direct", case=False, na=False) &
        df["Option"].str.contains("growth", case=False, na=False)
    ]
    df = df[(df["Date"] >= "2022-01-01") & (df["Date"] <= "2025-12-31")]
    df = df.dropna(subset=["Date", "NAV"]).sort_values("Date").reset_index(drop=True)

    if df.empty:
        raise ValueError(f"No usable Direct Growth rows found in {file_path}")

    df.insert(1, "Fund", fund_name)
    df["Daily_Return"] = df["NAV"].pct_change()
    df["Running_Max"] = df["NAV"].cummax()
    df["Drawdown"] = (df["NAV"] - df["Running_Max"]) / df["Running_Max"]
    return df


In [22]:
def calculate_cagr(df):
    start_nav = df["NAV"].iloc[0]
    end_nav = df["NAV"].iloc[-1]
    years = (df["Date"].iloc[-1] - df["Date"].iloc[0]).days / 365.25
    return (end_nav / start_nav) ** (1 / years) - 1


def calculate_volatility(df):
    return df["Daily_Return"].dropna().std() * np.sqrt(252)


def calculate_max_drawdown(df):
    return df["Drawdown"].min()


RISK_FREE_RATE = 0.06
TRADING_DAYS = 252

def calculate_sharpe(df, risk_free_rate=RISK_FREE_RATE):
    daily_returns = df["Daily_Return"].dropna()
    if len(daily_returns) < 2:
        return np.nan
    daily_rf = (1 + risk_free_rate) ** (1 / TRADING_DAYS) - 1
    daily_std = daily_returns.std()
    if pd.isna(daily_std) or daily_std == 0:
        return np.nan
    return ((daily_returns - daily_rf).mean() / daily_std) * np.sqrt(TRADING_DAYS)


In [23]:
clean_dir = Path("Mid_cap/clean")
analysis_dir = Path("Mid_cap/analysis")
clean_dir.mkdir(parents=True, exist_ok=True)
analysis_dir.mkdir(parents=True, exist_ok=True)

cleaned_data = {}
results = []
for fund_name, file_path in funds.items():
    df = clean_nav_data(file_path, fund_name)
    cleaned_data[fund_name] = df
    slug = fund_name.lower().replace(" ", "_").replace("midcap", "mid_cap")
    df.to_csv(clean_dir / f"{slug}.csv", index=False)
    df.to_csv(analysis_dir / f"{slug}_analysis.csv", index=False)

    results.append({
        "Fund": fund_name,
        "Start Date": df["Date"].min(),
        "End Date": df["Date"].max(),
        "Observations": len(df),
        "Start NAV": df["NAV"].iloc[0],
        "End NAV": df["NAV"].iloc[-1],
        "CAGR": calculate_cagr(df),
        "Volatility": calculate_volatility(df),
        "Maximum Drawdown": calculate_max_drawdown(df),
        "Sharpe Ratio": calculate_sharpe(df)
    })

mid_cap_results = pd.DataFrame(results)
display(mid_cap_results)


,Fund,Start Date,End Date,Observations,Start NAV,End NAV,CAGR,Volatility,Maximum Drawdown,Sharpe Ratio
0,Aditya Birla Sun Life Midcap Fund,2022-01-03,2025-12-31,985,521.9100,899.2600,0.146025,0.150345,-0.218746,0.614893
1,Axis Midcap Fund,2022-01-03,2025-12-31,990,79.0900,132.0700,0.137065,0.142198,-0.195165,0.580507
2,DSP Midcap Fund,2022-01-03,2025-12-31,985,98.9470,166.5210,0.139286,0.147503,-0.214060,0.582959
3,HDFC Mid Cap Fund,2022-01-03,2025-12-31,985,99.1800,224.7450,0.227437,0.148930,-0.167584,1.090738
4,Mahindra Manulife Mid Cap Fund,2022-01-03,2025-12-31,985,19.0035,38.4986,0.193467,0.161085,-0.209212,0.842090


In [24]:
results_path = analysis_dir / "mid_cap_fund_results.csv"
mid_cap_results.to_csv(results_path, index=False)
print(f"Fund-level results saved to: {results_path}")


Fund-level results saved to: Mid_cap/analysis/mid_cap_fund_results.csv
